In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
%run ./env

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

'current_catalog : mal_maite_bi_preprd'

'current_schema : process_time_analyses'

'current_location : abfss://maite_bi@adlsdpcompreprddata.dfs.core.windows.net/process_time_analyses/'

'common_schema : common'

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./translation_function

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

...

'current_schema : process_time_analyses'

'current_location : abfss://maite_bi@adlsdpcompreprddata.dfs.core.windows.net/process_time_analyses/'

'common_schema : common'

## `build_translation_dim`

Rend dense une table `*_translations` source. Voir `build_translations` pour le
détail du raisonnement ; en résumé, trois problèmes sont traités ici :

1. le **fallback** (langue → anglais → clé) n'est pas réalisable en RLS,
   qui sait supprimer des lignes mais pas en substituer une ;
2. les traductions sont **incomplètes** : une clé non traduite disparaîtrait du
   visuel après filtrage RLS, et les totaux seraient faux ;
3. le couple (clé, langue) doit être **unique** pour qu'il reste exactement une
   ligne par clé après filtrage.

## `publish_dim`

Écriture avec la mécanique Delta du projet, en `mode="full"` imposé.

En mode `update`, `handle_table_update` ne fait qu'insérer et mettre à jour : il
ne supprime jamais. Une clé retirée à la source resterait donc indéfiniment dans
la table du modèle et continuerait d'apparaître dans les slicers du rapport — un
libellé fantôme, sans données derrière. Ces tables sont petites et entièrement
redérivées à chaque run : le remplacement complet est plus sûr et sans coût
notable.

In [0]:
%run ./dim_trad_language

'Debug Mode'

'current_division : mal'

'current_environment : preprd'

'current_project : maite_bi'

'current_production_line : process_time_analyses'

...

Import dim_trad_language

mal_maite_bi_preprd.common.dim_trad_language


_1
language
code
language_label
culture_code


['code', 'language_label', 'culture_code']


Table mal_maite_bi_preprd.common.dim_trad_language has been fully replaced (delete + insert).
Number of rows inserted: 4


# build_translations

Construit les tables de traduction du modèle à partir des tables
`*_translations` de la base PostgreSQL.

## Pourquoi une transformation est nécessaire

Les tables source ne peuvent pas être branchées telles quelles sur le modèle :

1. **Le fallback demandé par le PO** (langue -> anglais -> la clé, jamais de null)
   n'est pas réalisable en RLS. Le RLS sait seulement *supprimer* des lignes, pas
   en substituer une autre. Le repli doit donc être matérialisé ici.
2. **Les traductions sont incomplètes** : toutes les langues ne sont pas
   renseignées pour toutes les clés. Sans traitement, une clé non traduite
   *disparaîtrait* du visuel après filtrage RLS — les lignes seraient absentes et
   **les totaux seraient faux**, ce qui est bien plus grave qu'un libellé anglais.
3. **Le couple (clé, langue) doit être unique et toujours présent** pour qu'après
   filtrage RLS il reste exactement une ligne par clé. Sinon les visuels
   dupliquent les lignes.

On produit donc des tables **denses** : le produit cartésien
`clés x langues actives`, avec un libellé garanti non nul sur chaque ligne.

## Chargement des sources

Ce notebook ne dépend pas de `load_data` : il tourne dans un job autonome qui
n'a pas besoin des tables de mesures. Il charge donc directement les 8 tables de
traduction depuis `source_catalog`, défini dans `env`.

In [0]:
goods_species_translations = spark.table(f"{source_catalog}.goods_species_translations")
goods_varieties_translations = spark.table(f"{source_catalog}.goods_varieties_translations")
parameters_production_type_translations = spark.table(f"{source_catalog}.parameters_production_type_translations")
parameters_variables_translations = spark.table(f"{source_catalog}.parameters_variables_translations")
parameters_production_line_variables_translations = spark.table(f"{source_catalog}.parameters_production_line_variables_translations")
parameters_localizations_translations = spark.table(f"{source_catalog}.parameters_localizations_translations")
parameters_localization_groups_translations = spark.table(f"{source_catalog}.parameters_localization_groups_translations")
parameters_batch_note_categories_translations = spark.table(f"{source_catalog}.parameters_batch_note_categories_translations")
processes_phases_translations = spark.table(f"{source_catalog}.processes_phases_translations")


## Traductions à clé simple

Ces cinq tables partagent le même patron : une clé entière, une langue, un
libellé. La clé est renommée pour correspondre à la colonne portée par la table
du modèle qui s'y rattachera.

## Les clés viennent des nomenclatures

La règle du PO — *« langue demandée, sinon anglais, sinon la clé ; jamais de
null »* — ne peut s'appliquer qu'à une ligne qui existe. Si les clés étaient
tirées des tables `*_translations`, une clé jamais traduite dans aucune langue
n'aurait aucune ligne, et son libellé s'afficherait **vide** dans le rapport :
le repli ne se déclencherait pas.

Les clés sont donc lues dans les nomenclatures de `common`, qui portent la liste
complète de ce qui existe réellement.

**Le repli de dernier recours est le `code` métier de la nomenclature**
(`ORGE_6RH`, `TCR`…) et non l'identifiant technique (`42`, `detail_TCR`) :
c'est l'interprétation la plus vraisemblable de « la clé » dans la bouche du PO,
et la seule qui produise quelque chose de lisible.

> Ce notebook doit s'exécuter **après** `build_nomenclatures`.
>
> Deux points restent à confirmer avec le PO : ce qu'il entend exactement par
> « la clé », et s'il a connaissance des clés jamais traduites (127 catégories
> de détail sur 397 en preprd).

In [0]:
def cles_nomenclature(table, key_col, alias, label_col=None):
    """Clés actives d'une nomenclature, renommées comme dans la table de traduction.

    label_col : la colonne de code métier, qui servira de repli de dernier recours.
    Les nomenclatures conservent les lignes supprimées (drapeau deleted) : on les
    écarte ici, une entité supprimée n'ayant pas à être traduite.
    """
    cols = [F.col(key_col).alias(alias)]
    if label_col is not None:
        cols.append(F.col(label_col).cast("string").alias(FALLBACK_LABEL_COL))
    else:
        # Pas de code métier dans cette nomenclature : on retombe sur l'identifiant.
        cols.append(F.col(key_col).cast("string").alias(FALLBACK_LABEL_COL))

    return (
        spark.table(f"{current_catalog}.{common_schema}.{table}")
        .filter(F.col("deleted") == False)
        .select(*cols)
    )

In [0]:
# goods_species_translations -> dim_trad_specy
dim_trad_specy = build_translation_dim(
    goods_species_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_specy", "id_good_specy", "good_specy", "code"),
    key_cols=["good_specy"]
).withColumnRenamed("good_specy", "id_good_specy")

publish_dim(dim_trad_specy, "dim_trad_specy", ["id_good_specy", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_specy -> clé ['id_good_specy', 'language'], colonnes ['id_good_specy', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_specy has been fully replaced (delete + insert).
Number of rows inserted: 16


'mal_maite_bi_preprd.common.dim_trad_specy'

In [0]:
# goods_varieties_translations -> dim_trad_variety
dim_trad_variety = build_translation_dim(
    goods_varieties_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variety", "id_good_variety", "good_variety", "code"),
    key_cols=["good_variety"]
).withColumnRenamed("good_variety", "id_good_variety")

publish_dim(dim_trad_variety, "dim_trad_variety", ["id_good_variety", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_variety -> clé ['id_good_variety', 'language'], colonnes ['id_good_variety', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_variety has been fully replaced (delete + insert).
Number of rows inserted: 264


'mal_maite_bi_preprd.common.dim_trad_variety'

In [0]:
# parameters_production_type_translations -> dim_trad_production_type
# La source nomme sa clé "id_parameters_production_type" (pluriel) alors que la
# table métier parameters_production_types porte "id_parameter_production_type"
# (singulier). On normalise ici sur le nom singulier, celui du modèle.
dim_trad_production_type = build_translation_dim(
    parameters_production_type_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_type", "id_parameter_production_type", "id_parameters_production_type", "code"),
    key_cols=["id_parameters_production_type"]
).withColumnRenamed("id_parameters_production_type", "id_parameter_production_type")

publish_dim(
    dim_trad_production_type, "dim_trad_production_type",
    ["id_parameter_production_type", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_production_type -> clé ['id_parameter_production_type', 'language'], colonnes ['id_parameter_production_type', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_production_type has been fully replaced (delete + insert).
Number of rows inserted: 160


'mal_maite_bi_preprd.common.dim_trad_production_type'

In [0]:
# parameters_variables_translations -> dim_trad_variable
dim_trad_variable = build_translation_dim(
    parameters_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_variable", "id_parameter_variable", "parameter_variable", "code"),
    key_cols=["parameter_variable"]
).withColumnRenamed("parameter_variable", "id_parameter_variable")

publish_dim(
    dim_trad_variable, "dim_trad_variable", ["id_parameter_variable", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_variable -> clé ['id_parameter_variable', 'language'], colonnes ['id_parameter_variable', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_variable has been fully replaced (delete + insert).
Number of rows inserted: 3808


'mal_maite_bi_preprd.common.dim_trad_variable'

In [0]:
# parameters_production_line_variables_translations -> dim_trad_production_line_variable
dim_trad_production_line_variable = build_translation_dim(
    parameters_production_line_variables_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_production_line_variable", "id_parameter_production_line_variable", "parameter_production_line_variable", None),
    key_cols=["parameter_production_line_variable"]
).withColumnRenamed(
    "parameter_production_line_variable", "id_parameter_production_line_variable"
)

publish_dim(
    dim_trad_production_line_variable, "dim_trad_production_line_variable",
    ["id_parameter_production_line_variable", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_production_line_variable -> clé ['id_parameter_production_line_variable', 'language'], colonnes ['id_parameter_production_line_variable', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_production_line_variable has been fully replaced (delete + insert).
Number of rows inserted: 13700


'mal_maite_bi_preprd.common.dim_trad_production_line_variable'

In [0]:
# parameters_localizations_translations -> dim_trad_localization
dim_trad_localization = build_translation_dim(
    parameters_localizations_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization", "id_parameter_localization", "id_parameter_localization", "code"),
    key_cols=["id_parameter_localization"]
)

publish_dim(
    dim_trad_localization, "dim_trad_localization",
    ["id_parameter_localization", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_localization -> clé ['id_parameter_localization', 'language'], colonnes ['id_parameter_localization', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_localization has been fully replaced (delete + insert).
Number of rows inserted: 80


'mal_maite_bi_preprd.common.dim_trad_localization'

## Groupes de localisation — clé ramenée à une seule colonne

`parameters_localization_groups_translations` porte une clé double
(`id_parameter_localization_group`, `production_line`) : le modèle *permet* qu'un
groupe ait un libellé différent selon la ligne de production.

**Vérifié en PROD : cette possibilité n'est pas utilisée.** Aucun couple
(groupe, langue) n'a plus d'un libellé distinct. On ramène donc la clé à
`id_parameter_localization_group` seul, ce qui aligne la table sur la
nomenclature `dim_localization_group` et donne une relation 1→\* propre dans
Power BI.

La cellule suivante contrôle cette hypothèse à chaque run : si le front se met
un jour à différencier les libellés par ligne de production, le job s'arrêtera
au lieu d'en retenir un au hasard.

In [0]:
# Contrôle : l'hypothèse "un seul libellé par (groupe, langue)" tient-elle ?
conflits_groupes = (
    parameters_localization_groups_translations
    .filter(F.col("deleted") == False)
    .groupBy("id_parameter_localization_group", "language")
    .agg(F.countDistinct("label").alias("nb_libelles"))
    .filter(F.col("nb_libelles") > 1)
)

if conflits_groupes.count() > 0:
    display(conflits_groupes)
    raise ValueError(
        "Des groupes de localisation ont plusieurs libellés selon la ligne de "
        "production. La clé de dim_trad_localization_group doit alors inclure "
        "production_line, et le modèle Power BI une colonne de substitution."
    )


dim_trad_localization_group = build_translation_dim(
    parameters_localization_groups_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_localization_group", "id_parameter_localization_group", "id_parameter_localization_group", "code"),
    key_cols=["id_parameter_localization_group"]
)

publish_dim(
    dim_trad_localization_group, "dim_trad_localization_group",
    ["id_parameter_localization_group", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_localization_group -> clé ['id_parameter_localization_group', 'language'], colonnes ['id_parameter_localization_group', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_localization_group has been fully replaced (delete + insert).
Number of rows inserted: 16


'mal_maite_bi_preprd.common.dim_trad_localization_group'

## Traductions des notes de production

`parameters_batch_note_categories_translations` sert **quatre usages** dans une
seule table : la clé `batch_note_category` est préfixée par le type
(`location_...`, `event_...`, `detail_...`, `impact_...`).

On en produit **quatre tables distinctes** plutôt qu'une seule. Raison :
`fact_batch_note` porte quatre colonnes à traduire (location, event, detail,
impact), et Power BI n'autorise qu'**une seule relation active** entre deux
tables. Une table unique obligerait à trois relations inactives et à des
`USERELATIONSHIP` dans chaque mesure — ingérable sur des colonnes posées
directement sur les axes des visuels.

Le repli de niveau 3 affiche le code **sans son préfixe** (`TCR` et non
`detail_TCR`) : c'est ce que l'utilisateur reconnaît.

In [0]:
# Le type de catégorie vient de la nomenclature (category_class), source faisant
# autorité, plutôt que d'un découpage du préfixe de la clé.
# La nomenclature conserve les noms source : sa clé est id_batch_note_category.
batch_note_categories = (
    spark.table(f"{source_catalog}.parameters_batch_note_categories")
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch_note_category").alias("batch_note_category"),
        F.col("category_class"),
        F.col("category_label"),
    )
)

batch_note_trad_base = (
    parameters_batch_note_categories_translations.alias("t")
    .join(
        F.broadcast(batch_note_categories).alias("n"),
        F.col("t.batch_note_category") == F.col("n.batch_note_category"),
        "inner"   # inner : une clé absente de la nomenclature n'a rien à traduire
    )
    .select("t.*", "n.category_class", "n.category_label")
)

if verbose_mode == 'debug':
    print("Traductions par type de catégorie :")
    display(
        batch_note_trad_base.filter(F.col("deleted") == False)
                            .groupBy("category_class").count().orderBy(F.desc("count"))
    )

Traductions par type de catégorie :


category_class,count
detail,1079
event,288
null,96
impact,88
location,76


In [0]:
BATCH_NOTE_TYPES = ["location", "event", "detail", "impact"]

for category_type in BATCH_NOTE_TYPES:
    df_type = batch_note_trad_base.filter(F.col("category_class") == category_type)

    dim_type = build_translation_dim(
        df_type,
        dim_trad_language,
        keys_df=cles_nomenclature(
            f"dim_batch_note_{category_type}",
            "id_batch_note_category", "batch_note_category",
            "category_label",
        ),
        key_cols=["batch_note_category"]
    )

    process_name = f"dim_trad_batch_note_{category_type}"
    globals()[process_name] = dim_type
    publish_dim(dim_type, process_name, ["batch_note_category", "language"], current_catalog + "." + common_schema)

mal_maite_bi_preprd.common.dim_trad_batch_note_location -> clé ['batch_note_category', 'language'], colonnes ['batch_note_category', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_batch_note_location has been fully replaced (delete + insert).
Number of rows inserted: 76
mal_maite_bi_preprd.common.dim_trad_batch_note_event -> clé ['batch_note_category', 'language'], colonnes ['batch_note_category', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_batch_note_event has been fully replaced (delete + insert).
Number of rows inserted: 288
mal_maite_bi_preprd.common.dim_trad_batch_note_detail -> clé ['batch_note_category', 'language'], colonnes ['batch_note_category', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_batch_note_detail has been fully replaced (delete + insert).
Number of rows inserted: 1080
mal_maite_bi_preprd.common.dim_trad_batch_note_impact -> clé ['batch_note_category', 'language'], 

## Contrôle qualité du rafraîchissement

`label_source` mesure la couverture réelle des traductions. Un taux élevé de
`fallback_key` sur une langue signale que le front n'a pas alimenté la table :
le rapport reste fonctionnel mais s'affiche en anglais ou en codes techniques.
C'est l'indicateur à remonter au PO, pas un incident technique.

## dim_trad_month

Les noms de mois ne viennent d'aucune table de traduction du front : ils sont
définis ici, une fois pour toutes. C'est un ensemble fermé de 13 clés (les
douze mois plus « non renseigné ») dans les 4 langues du projet.

Contrairement aux autres tables de traduction, il n'y a donc pas de repli à
prévoir : chaque clé a son libellé dans chaque langue, et `label_source` vaut
toujours `translated`.

Les identifiants de langue sont résolus depuis `dim_trad_language` par leur
`culture_code`, jamais codés en dur : ils diffèrent d'un environnement à
l'autre.


In [0]:
MOIS = {
    "fr": ["Non renseigné", "janvier", "février", "mars", "avril", "mai", "juin",
           "juillet", "août", "septembre", "octobre", "novembre", "décembre"],
    "en": ["Not specified", "January", "February", "March", "April", "May", "June",
           "July", "August", "September", "October", "November", "December"],
    "ro": ["Nespecificat", "ianuarie", "februarie", "martie", "aprilie", "mai", "iunie",
           "iulie", "august", "septembrie", "octombrie", "noiembrie", "decembrie"],
    "cs": ["Neuvedeno", "leden", "únor", "březen", "duben", "květen", "červen",
           "červenec", "srpen", "září", "říjen", "listopad", "prosinec"],
}

# (culture_code, month_num, label) : l'indice 0 porte le libellé « non renseigné ».
lignes_mois = [
    (culture, num, libelle)
    for culture, libelles in MOIS.items()
    for num, libelle in enumerate(libelles)
]

mois_par_culture = spark.createDataFrame(
    lignes_mois, "culture_code string, month_num int, label string"
)

# La jointure sur dim_trad_language garantit qu'on n'écrit que les langues
# réellement actives, et qu'on récupère leur identifiant tel qu'il existe dans
# cet environnement.
dim_trad_month = (
    mois_par_culture.alias("m")
    .join(F.broadcast(dim_trad_language).alias("l"), "culture_code", "inner")
    .select(
        F.col("m.month_num"),
        F.col("l.language"),
        F.col("m.label"),
        F.lit("translated").alias("label_source"),
    )
)

attendu = 13 * dim_trad_language.count()
obtenu = dim_trad_month.count()
if obtenu != attendu:
    raise ValueError(
        f"dim_trad_month : {obtenu} lignes au lieu de {attendu}. "
        "Une langue de dim_trad_language n'a pas ses noms de mois dans MOIS."
    )

publish_dim(
    dim_trad_month, "dim_trad_month",
    ["month_num", "language"], current_catalog + "." + common_schema)


mal_maite_bi_preprd.common.dim_trad_month -> clé ['month_num', 'language'], colonnes ['month_num', 'language', 'label', 'label_source']
Table mal_maite_bi_preprd.common.dim_trad_month has been fully replaced (delete + insert).
Number of rows inserted: 52


'mal_maite_bi_preprd.common.dim_trad_month'

## dim_trad_processes_phases

Les phases de process affichees par l'onglet VESSEL du rapport Self Service.

Le rapport lit aujourd'hui `gold.phases[prd_workshop]`, qui porte le code
technique anglais (`steeping`, `kilning`...). Cette table fournit le libelle
traduit correspondant.

> Le rapprochement se fera dans Power BI sur le **code**, pas sur
> l'identifiant : `gold.phases` ne porte pas `id_process_phase`. La colonne
> `code` est donc conservee ici, en plus de la cle, pour servir de pivot a la
> relation.


In [0]:
# processes_phases_translations -> dim_trad_processes_phases
dim_trad_processes_phases = build_translation_dim(
    processes_phases_translations,
    dim_trad_language,
    keys_df=cles_nomenclature("dim_processes_phases", "id_process_phase", "process_phase", "code"),
    key_cols=["process_phase"]
).withColumnRenamed("process_phase", "id_process_phase")

# Le code est rapatrie depuis la nomenclature : c'est lui qui porte la relation
# vers gold.phases[prd_workshop], l'identifiant n'etant pas present cote fait.
dim_trad_processes_phases = (
    dim_trad_processes_phases.alias("t")
    .join(
        spark.table(f"{current_catalog}.{common_schema}.dim_processes_phases")
             .filter(F.col("deleted") == False)
             .select(F.col("id_process_phase"), F.col("code")).alias("n"),
        "id_process_phase", "left",
    )
    .select("t.*", F.col("n.code"))
)

publish_dim(
    dim_trad_processes_phases, "dim_trad_processes_phases",
    ["id_process_phase", "language"], current_catalog + "." + common_schema)


mal_maite_bi_preprd.common.dim_trad_processes_phases -> clé ['id_process_phase', 'language'], colonnes ['id_process_phase', 'language', 'label', 'label_source', 'code']
Table mal_maite_bi_preprd.common.dim_trad_processes_phases créée (schéma seul, 0 ligne).
Table mal_maite_bi_preprd.common.dim_trad_processes_phases has been fully replaced (delete + insert).
Number of rows inserted: 36


'mal_maite_bi_preprd.common.dim_trad_processes_phases'

In [0]:
# Le contrôle relit les tables Delta qui viennent d'être écrites, il ne rejoue pas
# les DataFrames. Deux raisons :
#   - performance : les DataFrames ne sont pas en cache, les réutiliser ferait
#     recalculer toute la chaîne depuis PostgreSQL (dédoublonnage, crossJoin,
#     jointures) et doublerait le temps du notebook ;
#   - fiabilité : on mesure ce qui est réellement en base, pas ce qui aurait dû
#     y être écrit.
couverture = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .groupBy("language", "label_source")
             .count()
             .withColumn("table", F.lit(nom))
        for nom, target in tables_publiees
    ]
)

display(
    couverture.alias("c")
              .join(F.broadcast(dim_trad_language).alias("l"), "language", "left")
              .groupBy("table", "code")
              .pivot("label_source", ["translated", "fallback_en", "fallback_key"])
              .agg(F.sum("count"))
              .orderBy("table", "code")
)

table,code,translated,fallback_en,fallback_key
dim_trad_batch_note_detail,CZ,269,1,null
dim_trad_batch_note_detail,EN,270,null,null
dim_trad_batch_note_detail,FR,270,null,null
dim_trad_batch_note_detail,RO,270,null,null
dim_trad_batch_note_event,CZ,72,null,null
dim_trad_batch_note_event,EN,72,null,null
dim_trad_batch_note_event,FR,72,null,null
dim_trad_batch_note_event,RO,72,null,null
dim_trad_batch_note_impact,CZ,22,null,null
dim_trad_batch_note_impact,EN,22,null,null


## Audit : les clés jamais traduites

Elles sont désormais **présentes** dans le modèle, avec leur code métier en
repli — l'utilisateur ne voit plus de vide. Mais ce sont bien des traductions
manquantes, à remonter à l'équipe front.

`label_source = 'fallback_key'` les identifie : ni la langue demandée, ni
l'anglais. Une clé dans ce cas y est pour les quatre langues, c'est donc une
absence totale et non un trou ponctuel.

In [0]:
jamais_traduites = reduce(
    lambda a, b: a.unionByName(b),
    [
        spark.table(target)
             .filter(F.col("label_source") == "fallback_key")
             .select(
                 F.lit(nom).alias("table"),
                 F.col("label").alias("code_affiche"),
             )
             .distinct()
        for nom, target in tables_publiees
    ]
)
jamais_traduites.cache()

print("Clés sans aucune traduction, par table :")
display(jamais_traduites.groupBy("table").count().orderBy(F.desc("count")))

print("Détail, à transmettre à l'équipe front :")
display(jamais_traduites.orderBy("table", "code_affiche"))

Clés sans aucune traduction, par table :


table,count
dim_trad_production_line_variable,2550
dim_trad_variable,327
dim_trad_localization_group,4


Détail, à transmettre à l'équipe front :


table,code_affiche
dim_trad_localization_group,germination
dim_trad_localization_group,kilning
dim_trad_localization_group,pre-germination
dim_trad_localization_group,steeping
dim_trad_production_line_variable,1
dim_trad_production_line_variable,10
dim_trad_production_line_variable,100
dim_trad_production_line_variable,101
dim_trad_production_line_variable,10152
dim_trad_production_line_variable,10158
